# Train WEPR on answers you already have

The four published detectors each belong to one model, so scoring any other model means
training a detector for it. **This notebook is the training half alone.** You bring the
answers your model already produced; it validates them, labels them, fits a detector and
saves it.

Nothing here calls a model. **No GPU, no endpoint, no API key** — the whole notebook is a
few seconds of CPU work, and it runs offline.

That split is deliberate, because the two halves have completely different costs.
Generating and judging answers needs a model and takes hours; fitting on them takes
seconds. Keeping the answers on disk means you can refit at a different `k`, on `epr`
instead of `wepr`, or against better labels, without paying for generation again.

| You need | Where it comes from |
|---|---|
| A question pack, with gold answers | Your QA set — or [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) step 1 |
| The model's answers, with `top_logprobs` | Any generation run — or [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) phase B |
| A verdict per answer | Matching against the gold answer, an LLM judge, or a human pass |

If you have neither yet, [train_wepr_pipeline](train_wepr_pipeline.ipynb) produces both
against an OpenAI-compatible endpoint, and [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) produces them as `vllm run-batch` jobs.

> **The shipped answers are synthetic.** The 100 questions and their gold answers are real
> facts — the first is a real TriviaQA row, taken from that dataset's official repository —
> but **no model produced the answers in `responses_sample.jsonl` and no judge graded
> them.** A stand-in answers correctly, or -- for a little under half of them -- with a
> different, plausible wrong answer, and its per-token log-probabilities are drawn from a half-normal whose spread is wider when
> it is wrong — overlapping, so some wrong answers are confident and some right ones
> hesitant. Two rows carry a failed generation, as a real batch does.
>
> Everything the notebook *does* is therefore real — the validation, the parsing, the fit,
> the saved weights — but the score it reports describes that simulation, not any model.
> Point `RESPONSES` at your own run and every number becomes meaningful.

## The inputs

Three files, joined on `custom_id`.

**A verdict cannot precede the answer it judges** — it describes text a model produced —
which is why it is not a column of the question pack. But once the answers exist the
verdicts are part of the data, so they travel as their own file rather than being rederived
by every reader. The one shipped here was computed by matching (§3 shows the rule); yours might come from an LLM judge or a human pass, and drops in the same
way.

**`questions_sample.json`** — the question pack. `question_id` is the field that travels:
it becomes `custom_id` on the answers, and that is what the join pairs on.

| Field | Used for |
|---|---|
| `question` | The prompt that was sent |
| `question_id` | Identifies it; must be unique |
| `short_answer` | The gold answer, which the label is computed against |
| `answer_aliases` | Other answers that also count as correct |

**`responses_sample.jsonl`** — one generated answer per line, in the OpenAI Batch output
shape: exactly what `vllm run-batch` writes and what `scripts/train_detector.py` reads, so
a phase B output file works here unchanged.

```
{"custom_id": "tc_33",
 "response": {"status_code": 200,
              "body": { ...an OpenAI v1 ChatCompletion... }},
 "error": null}
```

Older `vllm` put the ChatCompletion directly in `response` with no envelope; both are
accepted below, as they are by the CLI.

**`judgments_sample.jsonl`** — one verdict per answer, in the *same* Batch output shape,
keyed by the same `custom_id`. The verdict is the judge's own reply, so the file is
literally what a judging run produces — nothing to convert:

```
{"custom_id": "tc_33",
 "response": {"body": {"choices": [{"message": {"content":
     "{\"judgment\": false, \"explanation\": \"...\"}"}}]}},
 "error": null}
```

**`judgment: true` means the generated answer was CORRECT**, which is the opposite of the
class the detector predicts. The conversion happens once, below: `hallucination = not
judgment`, so `1` marks a hallucination — the same convention as
`predict_proba(...)[:, 1]` and the `grounded` / `hallucination` classification report.

That key is the judge prompt's, not ours to rename: `scripts/ecir/prompts/judge.txt` asks
for it and `scripts/train_detector.py` parses it. Keeping it means these two files are the
ones [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) phase B writes, readable by the CLI as they stand.

In [1]:
# Colab, or any other kernel that is not this repository's environment: install what this
# notebook needs and fetch the three files it reads.
# A checkout that ran `uv sync` has both already, so nothing below runs there.
#
# uv rather than pip, because pip is the slow half of the wait: installing this package
# into an empty environment measured 17 s under pip, against 3 s to pip-install uv plus 1 s
# for uv to do the same work. On Colab, where most of the dependency tree is already
# present, the gap is smaller.
#
# Plain Python rather than the `!pip` and `%pip` magics, so the cell stays valid Python:
# the tests that run these notebooks compile the code cells, and so do the linters.
import importlib
import importlib.metadata
import subprocess
import sys
import urllib.request
from pathlib import Path


def missing(distribution):
    """Whether `distribution` is installed in the interpreter running this kernel.

    Asked of the installed distribution rather than of an import, because a bare directory
    named `artefactual` -- which is what cloning this repository beside the notebook leaves
    behind -- is an empty namespace package: `find_spec` finds it and `import artefactual`
    succeeds, so both would report the package present and skip the install. Only the
    metadata distinguishes a directory from a package.
    """
    try:
        importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return True
    return False


def shadowed(name):
    """Whether a directory beside the notebook hides an installed package of that name.

    The working directory comes first on `sys.path`, so such a directory wins over
    anything installed and the import fails on a submodule, several cells from the cause.
    """
    return Path(name).is_dir() and not Path(name, "__init__.py").exists()


def install(*packages):
    """Install into the interpreter running this kernel, showing what went wrong if it does.

    `-q` and no captured output is how an install failure becomes a bare
    `CalledProcessError` with the resolver's explanation nowhere on screen.
    """
    bootstrap = subprocess.run([sys.executable, "-m", "pip", "install", "-qU", "uv"], capture_output=True, text=True)
    resolve = [sys.executable, "-m", "uv", "pip", "install", "--python", sys.executable, "-q", *packages]
    done = bootstrap if bootstrap.returncode else subprocess.run(resolve, capture_output=True, text=True)
    if done.returncode:
        print(done.stderr or done.stdout)
        done.check_returncode()
    # The kernel started before these files existed, so the import machinery has a cached
    # listing of a directory that did not contain them.
    importlib.invalidate_caches()


if missing("artefactual"):
    install("artefactual")

assert not shadowed("artefactual"), (
    "a directory named 'artefactual' beside this notebook is hiding the installed "
    "package; rename it, or run this notebook from somewhere else"
)


RAW = "https://raw.githubusercontent.com/artefactory/artefactual/main/docs/examples/"


def fetch(name):
    """Download a file published beside this notebook, unless it is already here.

    Written to a temporary name and renamed, so an interrupted transfer leaves no
    half-written file for the `exists()` check to accept on the next run, and given a
    timeout so a black-holing proxy fails rather than hangs.
    """
    fixture = Path(name)
    if fixture.exists():
        return
    partial = fixture.with_suffix(fixture.suffix + ".part")
    with urllib.request.urlopen(RAW + name, timeout=30) as remote:
        partial.write_bytes(remote.read())
    partial.rename(fixture)


fetch("questions_sample.json")
fetch("responses_sample.jsonl")
fetch("judgments_sample.jsonl")

### What the detector actually reads

Not the answer text — the **token distribution behind it**. Inside that ChatCompletion,
one field is the entire input:

```
choices[0].logprobs.content[i].top_logprobs      # K ranks for token i  <-- this
choices[0].message.content                       # the answer, read only to label it
```

A response generated without `logprobs=True` is a perfectly valid completion carrying
nothing to score. One generated with **fewer than `K` ranks** is refused rather than
zero-filled, because the missing ranks are unfetched rather than absent — filling them
with zeros would drop their entropy and score the answer as more confident than it was.
Generating *wider* than `K` is fine; surplus ranks are dropped.

This is the single most common way a training run fails, so it is checked below rather
than left to surface inside `fit`.

In [2]:
import contextlib
import json
from collections import Counter
from pathlib import Path

QUESTIONS = Path("questions_sample.json")
RESPONSES = Path("responses_sample.jsonl")
JUDGMENTS = Path("judgments_sample.jsonl")

# Ranks per token. Part of the feature definition, not a batch size: WEPR fits one
# coefficient per rank, so the detector must later be loaded at the value it was fit at.
# Every published detector uses 15.
K = 15
SEED = 42

## 1. Load and validate the question pack

Every check here fails loudly rather than being repaired, because each one is a symptom of
the pack being built wrong, and a quietly-patched pack trains a detector nobody can
account for.

In [3]:
questions = json.loads(QUESTIONS.read_text(encoding="utf-8"))

REQUIRED = ("question", "question_id", "short_answer")
missing = [q for q in questions if any(not q.get(field) for field in REQUIRED)]
assert not missing, f"{len(missing)} question(s) missing one of {REQUIRED}, e.g. {missing[0]}"

# Duplicate ids would collapse in the join, pairing an answer with another question's gold
# answer -- which produces a wrong label rather than an error, so it is refused here.
duplicates = [qid for qid, n in Counter(q["question_id"] for q in questions).items() if n > 1]
assert not duplicates, f"question_id is not unique: {duplicates[:5]}"

by_id = {q["question_id"]: q for q in questions}
print(f"{len(questions)} questions, ids unique")
print(json.dumps(questions[0], indent=2))

100 questions, ids unique
{
  "question": "Which Lloyd Webber musical premiered in the US on 10th December 1993?",
  "question_id": "tc_33",
  "short_answer": "Sunset Boulevard",
  "answer_aliases": [
    "Sunset Blvd",
    "Sunset Blvd.",
    "Sunset Bulevard",
    "West Sunset Boulevard"
  ]
}


## 2. Load and validate the answers

Four things have to hold, and each is checked separately so a failure says which:

1. every line parses and carries a `custom_id`;
2. failed generations (`error` set, or a null response) are dropped and counted, not fed
   to the fit;
3. every answer joins to a question;
4. **every answer carries at least `K` ranks on every token** — not just the first one.
   Checking the first token only would pass a file whose later tokens are narrower, and
   the failure would then surface inside `fit` with no line number.

In [4]:
records, failed = [], 0
for number, line in enumerate(RESPONSES.read_text(encoding="utf-8").splitlines(), start=1):
    if not line.strip():
        continue
    # Without the line number, a malformed line reports "Expecting value: line 1 column 1"
    # -- about a line 1 that is in no file the reader has.
    try:
        record = json.loads(line)
    except json.JSONDecodeError as error:
        message = f"{RESPONSES} line {number} is not JSON: {error}"
        raise ValueError(message) from error
    assert "custom_id" in record, f"{RESPONSES} line {number} has no custom_id, so nothing can join it"

    # The Batch spec wraps the completion in {status_code, request_id, body}; older vllm
    # emitted it bare. Unwrap only when the envelope is there, as the CLI does.
    envelope = record["response"] or {}
    completion = envelope.get("body", envelope) if envelope else None
    # Three ways a line says it has no answer, and only the first is an `error`: a rejected
    # request comes back under a 4xx with an error object where the completion belongs, and
    # a failed one can carry an envelope with nothing in it. All three drop here, because
    # the alternative is a `NoneType` several cells away from the line that caused it.
    rejected = not 200 <= envelope.get("status_code", 200) < 300
    if record.get("error") is not None or rejected or not isinstance(completion, dict) or "choices" not in completion:
        failed += 1
        continue
    records.append((record["custom_id"], completion))

if failed:
    print(f"dropped {failed} failed generation(s)")

assert records, f"{RESPONSES} carried no usable responses"

# The pack was checked for duplicate ids; so is this file, and for the same reason. A
# repeated id here is worse, though: `records` is a list, so both copies survive, both get
# the same label, and `train_test_split` can put one in train and the other in test --
# which reads as a detector that generalises.
repeated = [cid for cid, count in Counter(cid for cid, _ in records).items() if count > 1]
assert not repeated, f"{len(repeated)} custom_id(s) appear on more than one response line, e.g. {repeated[:3]}"

unknown = [cid for cid, _ in records if cid not in by_id]
assert not unknown, f"{len(unknown)} answer(s) have no question with that id, e.g. {unknown[:3]}"

unanswered = sorted(set(by_id) - {cid for cid, _ in records})
if unanswered:
    print(f"{len(unanswered)} question(s) have no answer and are not trained on, e.g. {unanswered[:3]}")

print(f"{len(records)} answers, all joining to a question")

dropped 2 failed generation(s)
2 question(s) have no answer and are not trained on, e.g. ['q-017', 'q-062']
98 answers, all joining to a question


In [5]:
def rank_widths(completion):
    """Ranks per token, one count per generated token."""
    content = completion["choices"][0].get("logprobs") or {}
    return [len(token.get("top_logprobs") or []) for token in content.get("content") or []]


narrow, empty = [], []
for custom_id, completion in records:
    widths = rank_widths(completion)
    if not widths:
        empty.append(custom_id)
    elif min(widths) < K:
        narrow.append((custom_id, min(widths)))

assert not empty, (
    f"{len(empty)} response(s) carry no log-probabilities at all, e.g. {empty[:3]}. "
    f"That is what a provider returns when logprobs were not requested or are unsupported; "
    f"regenerate with logprobs=True and top_logprobs={K}."
)
assert not narrow, (
    f"{len(narrow)} response(s) are narrower than k={K}, e.g. {narrow[:3]} (id, narrowest token). "
    f"The missing ranks are unfetched rather than absent, so they cannot be zero-filled; "
    f"regenerate with top_logprobs={K}, or set K to the narrowest width and fit there."
)

widths = Counter(w for _, completion in records for w in rank_widths(completion))
print(f"rank widths across every token of every answer: {dict(widths)}")
print(f"all >= K = {K}")

rank widths across every token of every answer: {15: 152}
all >= K = 15


## 3. Load and validate the verdicts

The labels the classifier is fitted on, read from the file and joined on `custom_id`.

The verdict lives in the judge's `message.content` as JSON. Models wrap that in prose or
code fences often enough that a bare `json.loads` is unsafe, so this falls back to scanning
for the literal token — the same two-step `scripts/train_detector.py` uses, for the same
reason.

Two checks, for the two ways a verdict file goes wrong: a verdict for an answer that is
not here (the files came from different runs), and an answer with no verdict (a partial
labelling pass). The first is refused, the second drops that answer with a count — it
cannot be trained on either way, but only one of them means the files do not belong
together.

In [6]:
def read_judgment(completion):
    """True when the judge said the answer was correct, None when the reply is unreadable."""
    content = completion["choices"][0]["message"]["content"]
    # Only a real boolean counts. `bool("false")` is True, so a judge that emits the value
    # as a string -- which a loose JSON schema invites -- would mark every wrong answer
    # correct, silently, and the class balance would still look plausible.
    with contextlib.suppress(json.JSONDecodeError, KeyError, TypeError):
        verdict = json.loads(content)["judgment"]
        if isinstance(verdict, bool):
            return verdict

    lowered = (content if isinstance(content, str) else "").lower()
    if '"judgment": true' in lowered:
        return True
    if '"judgment": false' in lowered:
        return False
    return None


hallucinated, unreadable = {}, []
for line in JUDGMENTS.read_text(encoding="utf-8").splitlines():
    if not line.strip():
        continue
    row = json.loads(line)
    if row.get("error") is not None or row.get("response") is None:
        continue
    envelope = row["response"]
    judgment = read_judgment(envelope.get("body", envelope))
    if judgment is None:
        unreadable.append(row["custom_id"])
        continue
    # `judgment: true` means the answer was CORRECT, so the label is its negation --
    # the one place the judge's convention and the detector's meet, hence the name.
    hallucinated[row["custom_id"]] = not judgment

if unreadable:
    print(f"dropped {len(unreadable)} verdict(s) that could not be parsed, e.g. {unreadable[:3]}")

answered = {custom_id for custom_id, _ in records}
stray = sorted(set(hallucinated) - answered)
assert not stray, (
    f"{len(stray)} verdict(s) judge an answer that is not in {RESPONSES.name}, e.g. {stray[:3]}. "
    f"The two files are from different runs."
)

unlabelled = sorted(answered - set(hallucinated))
if unlabelled:
    print(f"dropped {len(unlabelled)} answer(s) with no verdict, e.g. {unlabelled[:3]}")

annotated = [
    (by_id[custom_id], completion, int(hallucinated[custom_id]))
    for custom_id, completion in records
    if custom_id in hallucinated
]

# The question travels with its answer and its label, so nothing downstream is indexed by
# position in another list.
responses = [completion for _, completion, _ in annotated]
labels = [label for _, _, label in annotated]
print(f"joined {len(annotated)} answers to their verdicts on custom_id")

joined 98 answers to their verdicts on custom_id


### Where these verdicts came from

Matching, in this sample's case: an answer counted as correct when it contained the gold
answer or one of its aliases, after both sides were normalised the way TriviaQA's own
metric normalises. That is a proxy and it is wrong in a predictable direction -- *"the
Billy Wilder adaptation"* is a correct answer for *Sunset Boulevard* and does not match --
so a judge model or a human pass produces better labels for the same answers. Because the
verdicts are a file, swapping them means replacing it and rerunning from here; the
generation is not repeated. Grading properly is what [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir)'s judge does.

In [7]:
for question, completion, label in annotated[:3]:
    print(f"[{'hallucination' if label else 'grounded'}] {question['question'][:60]}")
    print(f"    gold: {question['short_answer']}")
    print(f"    said: {completion['choices'][0]['message']['content'].strip()[:80]}\n")

[hallucination] Which Lloyd Webber musical premiered in the US on 10th Decem
    gold: Sunset Boulevard
    said: Cats

[hallucination] Who wrote the novel 'Things Fall Apart'?
    gold: Chinua Achebe
    said: Wole Soyinka

[grounded] What is the capital of Mongolia?
    gold: Ulaanbaatar
    said: Ulaanbaatar



In [8]:
import numpy as np

y = np.array(labels)

# Both classes are needed, and enough of the rarer one to sit on both sides of the split.
# This is where a run fails cheaply rather than inside `fit` or, worse, inside a score
# computed over a held-out set with one class in it. All-correct means the questions were
# too easy for this model; all-wrong usually means it is not answering in the short form
# the verdicts expect.
rarer = min(y.sum(), len(y) - y.sum())
assert rarer >= 5, (
    f"only {rarer} answer(s) in the rarer class out of {len(y)}; a holdout cannot say anything "
    f"at that size. Label more answers, or make the questions harder or easier."
)

print(f"{len(y)} labelled answers, {y.sum()} hallucinations ({y.mean():.0%})")

98 labelled answers, 47 hallucinations (48%)


## 4. Fit, and score on held-out answers

`trainable=True` returns an unfitted pipeline — parser, entropy reduction, logistic
regression — that takes the raw responses, so there is no feature extraction to write. It
is explicit by design: calling `wepr()` with neither weights nor `trainable=True` raises,
rather than handing back a detector that would emit probabilities no trained weights
support.

The split is stratified and happens before the fit, so what is reported describes answers
the detector never saw. **ROC-AUC** scores the ranking, which governs triage by score and
is what the paper reports; the **classification report** scores the decisions at 0.5,
where recall on the `hallucination` row is the fraction actually flagged. Only the AUC
carries over to another threshold, so pick a threshold from these scores rather than
assuming 0.5 — [the scoring
guide](https://artefactory.github.io/artefactual/guide/scoring.html) covers how.

A holdout of a few dozen answers shows whether the signal is there and cannot pin it down.
More labelled answers is the only fix.

In [9]:
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

from artefactual.scoring import wepr

x_train, x_test, y_train, y_test = train_test_split(responses, y, test_size=0.25, stratify=y, random_state=SEED)
detector = wepr(k=K, trainable=True).fit(x_train, y_train)
print(f"fitted on {len(y_train)}, holding out {len(y_test)}")

scores = detector.predict_proba(x_test)[:, 1]
# Two decimals, because a holdout of this size cannot support a third: refitting the same
# data under different split seeds moves this number by tenths. `scripts/train_detector.py`
# reports a bootstrap interval around it, which is the honest form of the same measurement.
print(f"\nROC-AUC: {roc_auc_score(y_test, scores):.2f}")
print(classification_report(y_test, scores >= 0.5, target_names=["grounded", "hallucination"], zero_division=0))

fitted on 73, holding out 25

ROC-AUC: 0.84
               precision    recall  f1-score   support

     grounded       0.91      0.77      0.83        13
hallucination       0.79      0.92      0.85        12

     accuracy                           0.84        25
    macro avg       0.85      0.84      0.84        25
 weighted avg       0.85      0.84      0.84        25



### Choosing a threshold

The report above decides at 0.5 because something has to. That is where the sigmoid
crosses, not where your costs balance: a detector that flags 90% of hallucinations at the
price of some false alarms is a different tool from one that only flags what it is certain
of, and both come from these same weights. `precision_recall_curve(y_test, scores)` gives
you the trade-off to pick from, and the held-out set is where to pick it.

**Not demonstrated here, because this sample cannot show it honestly.** The classifier is
unregularised (`C=inf`) and these synthetic classes are close to separable, so it saturates:
most of the scores round to 0.0 or 1.0 and the curve has very little in between. On real
answers the scores spread out and the choice becomes a real one.

### `epr` instead of `wepr`

The same call with one word changed. EPR pools every rank into a single feature; WEPR keeps
one coefficient per rank, `2k` of them. Everything else -- the parser, the data, the fit --
is identical.

In [10]:
from artefactual.scoring import epr

for name, factory in (("epr", epr), ("wepr", wepr)):
    fitted = factory(k=K, trainable=True).fit(x_train, y_train)
    width = fitted.named_steps["classifier"].coef_.shape[1]
    print(f"{name:>5}: {width:>2} coefficient{'s' if width > 1 else ''} for k={K}")

  epr:  1 coefficient for k=15
 wepr: 30 coefficients for k=15


Which of the two scores better is a question about your model and your data, and this
sample cannot answer it: the log-probabilities are simulated and the fit sees 73 answers.
The paper measured it properly -- WEPR beat EPR on every model it tested, which is why
`wepr` is the default -- and [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) reproduces those numbers.

## 5. Save the weights, and use them

The same `.skops` format the published detectors ship in, loaded through the same call — a
repository id, a path, either one. `k` must be the value the weights were fitted at;
another value raises rather than mis-shaping the score.

From here it is an ordinary detector. `predict_proba` gives one score per response;
`predict_token_proba` gives one per token, which
[wepr_usage_demo](wepr_usage_demo.ipynb) shows properly — the answers in this sample are
one to four tokens long, so there would be nothing to see.

To publish it the way the shipped detectors are published, the file has to be named
`model.skops`, which is what `wepr("me/my-detector")` looks for on the Hub:
`save_estimator("my-detector/model.skops")`, creating the directory on the way. Passing
just `my-detector/` writes a *file* of that name unless the directory already exists —
`Path` drops the trailing slash, and the directory check is made on what is there.

In [11]:
path = detector.save_estimator("wepr-trained.skops")
reloaded = wepr(path, k=K)

# Held-out answers, not the first rows of `annotated`: most of those were fitted on, and
# scoring them would show how well the detector memorised rather than how it generalises.
for completion, label in list(zip(x_test, y_test))[:5]:
    probability = reloaded.predict_proba(completion)[0, 1]
    marker = "hallucination" if label else "grounded    "
    said = completion["choices"][0]["message"]["content"].strip()
    print(f"[{marker}] P={probability:.3f}  said {said[:40]!r}")

[hallucination] P=1.000  said 'Saturn'
[hallucination] P=1.000  said 'Colombia'
[hallucination] P=1.000  said 'David Ricardo'
[grounded    ] P=1.000  said 'Ottawa'
[grounded    ] P=0.000  said 'Beethoven'


## Where to go next

- **More labelled answers.** The fit is seconds — the cost is entirely in producing the
  answers. Refitting at a different `k` or on `epr` costs nothing, because it reads the
  same file.
- **Produce the answers.** [train_wepr_pipeline](train_wepr_pipeline.ipynb) generates and
  judges them against an OpenAI-compatible endpoint; [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir) runs both stages as
  `vllm run-batch` jobs, which is the practical way at thousands of questions.
- **Better labels.** Matching calls a correct-but-reworded answer a hallucination. The
  paper's LLM judge, in [`scripts/ecir`](https://github.com/artefactory/artefactual/tree/main/scripts/ecir), reads those cases correctly — and because the answers are
  a file, relabelling never means regenerating.
- **`epr` instead of `wepr`.** Same call, same data, one feature instead of `2k`. WEPR beat
  EPR on every row of the paper's table, which is why it is the default here.